In [6]:
import requests
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer, util
import torch


class SemanticSearchEngine:

    def __init__(self):
        self.model = SentenceTransformer(
            "intfloat/e5-large-v2"
        )

        self.chunks = []
        self.metadata = []
        self.embeddings = None

    def extract_text_from_url(self, url):

        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()

            soup = BeautifulSoup(
                response.text,
                "html.parser"
            )

            for tag in soup(["script", "style"]):
                tag.decompose()

            text = soup.get_text(separator=" ")

            text = " ".join(text.split())

            return text

        except Exception as e:
            print(f"Error scraping {url}: {e}")
            return ""

    def chunk_text(
        self,
        text,
        chunk_size=500,
        overlap=100
    ):
        chunks = []

        start = 0

        while start < len(text):

            end = start + chunk_size

            chunk = text[start:end]

            chunks.append(chunk)

            start += chunk_size - overlap

        return chunks

    def index_urls(self, urls):

        all_chunks = []
        all_metadata = []

        for url in urls:

            print(f"Processing: {url}")

            text = self.extract_text_from_url(url)

            if not text:
                continue

            chunks = self.chunk_text(text)

            for chunk in chunks:

                all_chunks.append(
                    f"passage: {chunk}"
                )

                all_metadata.append(
                    {
                        "url": url,
                        "text": chunk
                    }
                )

        print(
            f"Generated {len(all_chunks)} chunks"
        )

        embeddings = self.model.encode(
            all_chunks,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )

        self.chunks = all_chunks
        self.metadata = all_metadata
        self.embeddings = embeddings

        print(
            f"Indexed {len(all_chunks)} chunks"
        )

    def search(
        self,
        query,
        top_k=5
    ):

        query_embedding = self.model.encode(
            f"query: {query}",
            convert_to_tensor=True,
            normalize_embeddings=True
        )

        scores = util.cos_sim(
            query_embedding,
            self.embeddings
        )[0]

        top_results = torch.topk(
            scores,
            k=min(top_k, len(scores))
        )

        results = []

        for score, idx in zip(
            top_results.values,
            top_results.indices
        ):

            results.append(
                {
                    "score": float(score),
                    "url": self.metadata[idx]["url"],
                    "text": self.metadata[idx]["text"]
                }
            )

        return results

In [8]:
urls = [
        "https://www.pinecone.io/learn/hybrid-search-intro/",
        "https://www.pinecone.io/blog/hybrid-search/"
    ]

engine = SemanticSearchEngine()

engine.index_urls(urls)

query = "What is hybrid search?"

results = engine.search(
    query,
    top_k=3
)

print("\n" + "=" * 80)
print("SEARCH RESULTS")
print("=" * 80)

for i, result in enumerate(results, start=1):

    print(f"\nResult #{i}")
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['url']}")
    print("-" * 50)

    print(result["text"])

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Processing: https://www.pinecone.io/learn/hybrid-search-intro/
Processing: https://www.pinecone.io/blog/hybrid-search/
Generated 67 chunks


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Indexed 67 chunks

SEARCH RESULTS

Result #1
Score: 0.8721
Source: https://www.pinecone.io/learn/hybrid-search-intro/
--------------------------------------------------
hybrid search. Hybrid Search Combining dense and sparse search takes work. In the past, engineering teams needed to run different solutions for dense and sparse search engines and another system to combine results in a meaningful way. Typically a dense vector index, sparse inverted index, and reranking step. The Pinecone approach to hybrid search uses a single sparse-dense index. It enables search across any modality; text, audio, images, etc. Finally, the weighting of dense vs. sparse can be ch

Result #2
Score: 0.8639
Source: https://www.pinecone.io/blog/hybrid-search/
--------------------------------------------------
izers. Consistency : Gives you the benefits of tried-and-true BM25 scoring for the keyword part of hybrid search. Control : Adjust the weight of keyword and semantic relevance when querying. Gives you c